In [1]:
from google.colab import files
import pathlib
import pickle
import warnings

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                               precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings("ignore")

DATA_PATH = pathlib.Path("/content/revised_master_dts_weekly.csv")
OUT_DIR   = pathlib.Path("/content/hab_outputs_xgb_weekly")
OUT_DIR.mkdir(exist_ok=True)

TARGET_SOURCE   = "bloom_or_documented"
TRAIN_END_YEAR  = 2023
N_ROUNDS_MAX    = 2000
EARLY_STOP      = 50
N_CV_SPLITS     = 4
CV_TEST_SIZE    = 46

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"  raw rows: {len(df)}, columns: {len(df.columns)}")

df = df.sort_values(["region", "year", "doy_start"]).reset_index(drop=True)
df = df.dropna(subset=["chlor_a_mean"]).copy()

  # 1- and 2-WEEK lag features
lag_cols = [
      "sst_mean", "sst_min", "sst_max", "sst_std",
      "rainfall_mm_total_mean", "rainfall_mm_max_daily", "rainy_days_mean",
  ]
for c in lag_cols:
     for lag in (1, 2):
        df[f"{c}_lag{lag}"] = df.groupby("region")[c].shift(lag)

  # Rolling means — 4 and 8 WEEK windows
roll_cols = ["sst_mean", "sst_std", "rainfall_mm_total_mean", "rainy_days_mean"]
for c in roll_cols:
    df[f"{c}_roll4w"] = df.groupby("region")[c].shift(1).rolling(4).mean().reset_index(0, drop=True)
    df[f"{c}_roll8w"] = df.groupby("region")[c].shift(1).rolling(8).mean().reset_index(0, drop=True)

  # Anomalies
anom_cols = ["sst_mean", "chlor_a_mean", "rainfall_mm_total_mean"]
for c in anom_cols:
    clim = df.groupby(["region", "doy_start"])[c].transform("mean")
    df[f"{c}_anomaly"] = df[c] - clim

  # Cumulative rainfall
df["rainfall_cum_4w"] = (
    df.groupby("region")["rainfall_mm_total_mean"]
      .shift(1).rolling(4).sum().reset_index(0, drop=True))
df["rainfall_cum_8w"] = (
    df.groupby("region")["rainfall_mm_total_mean"]
      .shift(1).rolling(8).sum().reset_index(0, drop=True))

  # SST slope
df["sst_slope_4w"] = (
    df.groupby("region")["sst_mean"].shift(1)
      - df.groupby("region")["sst_mean"].shift(5)) / 4

  # Cyclic day-of-year + region
df["window_sin"] = np.sin(2 * np.pi * df["doy_start"] / 365)
df["window_cos"] = np.cos(2 * np.pi * df["doy_start"] / 365)
df["region_Kerala"]    = (df["region"] == "Kerala").astype(int)
df["region_Karnataka"] = (df["region"] == "Karnataka").astype(int)

FEATURES = (
      lag_cols
      + [f"{c}_lag{l}" for c in lag_cols for l in (1, 2)]
      + [c for c in df.columns if "_roll" in c or "_anomaly" in c
                                or "_cum_" in c or "_slope_" in c]
      + ["window_sin", "window_cos", "region_Kerala", "region_Karnataka"]
      + ["hab_event_documented", "hab_events_last_52w"]
  )
FEATURES = list(dict.fromkeys(FEATURES))
print(f"  total features: {len(FEATURES)}")

  raw rows: 460, columns: 25
  total features: 41


In [3]:
df["bloom_next_week"] = df.groupby("region")[TARGET_SOURCE].shift(-1)
TARGET = "bloom_next_week"

df_model = df.dropna(subset=FEATURES + [TARGET]).copy().sort_values(
      ["year", "doy_start", "region"]).reset_index(drop=True)

train = df_model[df_model["year"] <= TRAIN_END_YEAR]
test  = df_model[df_model["year"] >  TRAIN_END_YEAR]

X_train, y_train = train[FEATURES], train[TARGET].astype(int)
X_test,  y_test  = test[FEATURES],  test[TARGET].astype(int)

print(f"  usable rows: {len(df_model)}")
print(f"  train: {len(X_train)} rows (positive rate {y_train.mean():.3f})")
print(f"  test:  {len(X_test)} rows (positive rate {y_test.mean():.3f})")

  usable rows: 211
  train: 187 rows (positive rate 0.310)
  test:  24 rows (positive rate 0.125)


In [4]:
X_cv = X_train.reset_index(drop=True)
y_cv = y_train.reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS, test_size=CV_TEST_SIZE)

fold_metrics = []
for fold, (tr_idx, te_idx) in enumerate(tscv.split(X_cv), 1):
     X_tr, y_tr = X_cv.iloc[tr_idx], y_cv.iloc[tr_idx]
     X_te, y_te = X_cv.iloc[te_idx], y_cv.iloc[te_idx]

scale_pos_w = (y_tr == 0).sum() / max(1, (y_tr == 1).sum())
fold_model = XGBClassifier(
          n_estimators=500, max_depth=5, learning_rate=0.05,
          subsample=0.8, colsample_bytree=0.8,
          scale_pos_weight=scale_pos_w,
          eval_metric="logloss",
          random_state=42, n_jobs=-1,
      )
fold_model.fit(X_tr, y_tr)
p = fold_model.predict_proba(X_te)[:, 1]
pred = (p >= 0.5).astype(int)

row = {"fold": fold, "train_n": len(tr_idx), "test_n": len(te_idx),
             "accuracy": accuracy_score(y_te, pred),
             "roc_auc":  roc_auc_score(y_te, p) if y_te.nunique() > 1 else float("nan")}
fold_metrics.append(row)
print(f"fold {fold}: train={row['train_n']}, test={row['test_n']}, "
            f"acc={row['accuracy']:.3f}, AUC={row['roc_auc']:.3f}")

cv_df = pd.DataFrame(fold_metrics)
print(f"\n== CV summary ({N_CV_SPLITS} folds) ==")
print(f"  accuracy: {cv_df.accuracy.mean():.3f} ± {cv_df.accuracy.std():.3f}")
print(f"  ROC-AUC:  {cv_df.roc_auc.mean():.3f} ± {cv_df.roc_auc.std():.3f}")

fold 4: train=141, test=46, acc=0.717, AUC=0.869

== CV summary (4 folds) ==
  accuracy: 0.717 ± nan
  ROC-AUC:  0.869 ± nan


In [5]:
train_sorted = train.sort_values(["year","doy_start","region"]).reset_index(drop=True)
cut = int(len(train_sorted) * 0.80)
Xtr = train_sorted.iloc[:cut][FEATURES]
ytr = train_sorted.iloc[:cut][TARGET].astype(int)
Xva = train_sorted.iloc[cut:][FEATURES]
yva = train_sorted.iloc[cut:][TARGET].astype(int)
scale_pos_w = (ytr == 0).sum() / max(1, (ytr == 1).sum())

model = XGBClassifier(
      n_estimators           = N_ROUNDS_MAX,
      max_depth              = 5,
      learning_rate          = 0.05,
      subsample              = 0.8,
      colsample_bytree       = 0.8,
      scale_pos_weight       = scale_pos_w,
      eval_metric            = "logloss",
      early_stopping_rounds  = EARLY_STOP,
      random_state           = 42,
      n_jobs                 = -1,
  )
model.fit(Xtr, ytr, eval_set=[(Xtr, ytr), (Xva, yva)], verbose=False)

best_iter = int(model.best_iteration)
print(f"early-stopped at round {best_iter} / {N_ROUNDS_MAX}")
print(f"val log-loss at best round: {model.best_score:.4f}")

proba = model.predict_proba(X_test)[:, 1]
pred  = (proba >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
metrics = {
      "accuracy":  accuracy_score(y_test, pred),
      "precision": precision_score(y_test, pred, zero_division=0),
      "recall":    recall_score(y_test, pred, zero_division=0),
      "f1":        f1_score(y_test, pred, zero_division=0),
      "roc_auc":   roc_auc_score(y_test, proba),
  }

print("\n=== FINAL HELD-OUT METRICS (7-day forecast, 2024) ===")
for k, v in metrics.items():
    print(f"  {k:>9s}: {v:.4f}")
print(f"  confusion matrix: TN={tn}  FP={fp}  FN={fn}  TP={tp}")

early-stopped at round 57 / 2000
val log-loss at best round: 0.3896

=== FINAL HELD-OUT METRICS (7-day forecast, 2024) ===
   accuracy: 0.9167
  precision: 0.6000
     recall: 1.0000
         f1: 0.7500
    roc_auc: 0.9524
  confusion matrix: TN=19  FP=2  FN=0  TP=3


In [6]:
with open(OUT_DIR / "XGBoost_weekly.pkl", "wb") as f:
      pickle.dump(model, f)

test[["year", "doy_start", "date_start", "region", "chlor_a_mean", TARGET]].assign(
      proba=proba, pred=pred
  ).to_csv(OUT_DIR / "XGB_weekly_predictions.csv", index=False)

imp = pd.DataFrame({
      "feature":    FEATURES,
      "importance": model.feature_importances_,
  }).sort_values("importance", ascending=False)
imp.to_csv(OUT_DIR / "XGB_weekly_feature_importance.csv", index=False)

cv_df.to_csv(OUT_DIR / "XGB_weekly_cv_folds.csv", index=False)

  # Training history — the Keras-style plot data
history = model.evals_result_
pd.DataFrame({
      "boosting_round": range(len(history["validation_0"]["logloss"])),
      "train_loss":     history["validation_0"]["logloss"],
      "val_loss":       history["validation_1"]["logloss"],
  }).to_csv(OUT_DIR / "XGB_weekly_training_history.csv", index=False)

pd.DataFrame([{
      "model": "XGBoost_weekly",
      "best_iteration": best_iter,
      "n_features": len(FEATURES),
      "cv_accuracy_mean": round(cv_df.accuracy.mean(), 4),
      "cv_accuracy_std":  round(cv_df.accuracy.std(),  4),
      "cv_roc_auc_mean":  round(cv_df.roc_auc.mean(),  4),
      "cv_roc_auc_std":   round(cv_df.roc_auc.std(),   4),
      **{f"heldout_{k}": round(v, 4) for k, v in metrics.items()},
      "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
  }]).to_csv(OUT_DIR / "XGB_weekly_metrics.csv", index=False)

print("=== TOP 10 FEATURES ===")
print(imp.head(10).to_string(index=False))

print("\nsaved:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")

import shutil
from google.colab import files
shutil.make_archive("xgb_weekly_outputs", "zip", OUT_DIR)
files.download("xgb_weekly_outputs.zip")

=== TOP 10 FEATURES ===
                      feature  importance
                sst_mean_lag2    0.095130
                   window_sin    0.048930
         rainy_days_mean_lag2    0.046957
rainfall_mm_total_mean_roll8w    0.045054
              rainfall_cum_8w    0.043081
   rainfall_mm_max_daily_lag2    0.043026
  rainfall_mm_total_mean_lag2    0.041295
                sst_mean_lag1    0.039405
         rainy_days_mean_lag1    0.035611
              sst_mean_roll8w    0.032124

saved:
  XGB_weekly_cv_folds.csv
  XGB_weekly_feature_importance.csv
  XGB_weekly_metrics.csv
  XGB_weekly_predictions.csv
  XGB_weekly_training_history.csv
  XGBoost_weekly.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>